# Agent 3: Advanced Risk Intelligence & ML

## Context
The **Risk Intelligence Agent** is the core actuarial engine of CRIP. It calculates multi-dimensional risk scores, trains predictive Machine Learning models to forecast claim severity, and runs Monte Carlo simulations to estimate Capital Adequacy.

## Working Logic
1. **Multi-Dimensional Risk Scoring:** Calculates individual scores (1-10 scale) for Insurance, Credit, Market, Operational, and Catastrophe (CAT) risks using weighted heuristics from `config.py`.
2. **Predictive Modeling (XGBoost):**
   - **Regressor:** Trains an XGBoost model to predict expected claim amounts.
   - **Classifier:** Trains an XGBoost model to predict the probability of a policy becoming "High Risk" (Loss Ratio > 1.0).
3. **Capital Adequacy (Monte Carlo):** Simulates thousands of loss scenarios to calculate the 99% Value at Risk (VaR) and Expected Shortfall.

## Key Concepts
- **XGBoost:** Extreme Gradient Boosting, a powerful ensemble learning method used here for robust regression and classification of risk profiles.
- **Value at Risk (VaR):** A statistical technique used to quantify the level of financial risk within a firm or investment portfolio over a specific time frame. A 99% VaR represents the maximum expected loss with a 99% confidence level.
- **Expected Shortfall (ES):** Also known as Conditional VaR, it estimates the expected loss given that the loss has exceeded the VaR threshold (the "tail risk").


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, brier_score_loss

### 1. Load Data
We load the output from the Pricing Agent (or a raw dataset).

In [ ]:
# Assuming df is loaded from the previous step
# df = pd.read_csv('reports/pricing_output.csv')

# For demonstration, we'll create a dummy dataframe if you just run this cell
df = pd.DataFrame({'Policy_ID': [1,2], 'Sum_Insured': [100000, 200000], 'Written_Premium': [1000, 2000], 'Claim_Amount': [500, 0], 'Loss_Ratio': [0.5, 0]})
print('Data Loaded.')

In [ ]:
def normalize(series):
    '''Min-Max normalization to 0-1 range'''
    series = pd.to_numeric(series, errors='coerce').fillna(0)
    if series.max() == series.min():
        return pd.Series(0, index=series.index)
    return (series - series.min()) / (series.max() - series.min())

### 2. Actuarial Risk Formulas (1-10)
Calculate Insurance, Credit, Market, Operational, and CAT risks using defined formulas.

In [ ]:
# 1. Insurance Risk
if 'Claim_Count' not in df.columns: df['Claim_Count'] = np.random.poisson(1, len(df))
if 'Exposure_At_Risk' not in df.columns: df['Exposure_At_Risk'] = df['Sum_Insured']

df['Claim_Frequency'] = df['Claim_Count'] / df['Exposure_At_Risk'].replace(0, 1)
df['Claim_Severity'] = df['Claim_Amount'] / df['Claim_Count'].replace(0, 1)

df['Insurance_Risk'] = (
    0.4 * normalize(df['Loss_Ratio'].fillna(0)) + 
    0.3 * normalize(df['Claim_Frequency']) + 
    0.3 * normalize(df['Claim_Severity'])
) * 10

# 2. Market & Credit Risk (Dummy inputs for demo)
df['Premium_Outstanding'] = np.random.uniform(0, 5000, len(df))
df['Days_Past_Due'] = np.random.randint(0, 90, len(df))
df['Credit_Risk'] = (normalize(df['Premium_Outstanding']) * 0.6 + normalize(df['Days_Past_Due']) * 0.4) * 10

# ... Operational and CAT Risk ...
df['Fraud_Score'] = np.random.uniform(0, 100, len(df))
df['Operational_Risk'] = (0.35 * normalize(df['Fraud_Score'])) * 10

df[['Insurance_Risk', 'Credit_Risk', 'Operational_Risk']].head()

### 3. XGBoost Predictive Pricing
Train a model to predict `Expected_Claim_Amount_ML` based on features.

In [ ]:
features = ['Sum_Insured', 'Premium_Outstanding', 'Insurance_Risk', 'Credit_Risk', 'Operational_Risk']
X = df[features].fillna(0)
y_claim = df['Claim_Amount'].fillna(0)

xgb_reg = xgb.XGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42)
xgb_reg.fit(X, y_claim)
df['Expected_Claim_Amount_ML'] = xgb_reg.predict(X)

print("XGBoost trained successfully!")

### 4. Monte Carlo Simulation (VaR & Solvency)
Simulate 1000 portfolio outcomes to find the 99% Value at Risk.

In [ ]:
num_simulations = 1000
mean_loss = df['Claim_Amount'].mean()
std_loss = df['Claim_Amount'].std() if pd.notna(df['Claim_Amount'].std()) and df['Claim_Amount'].std() > 0 else mean_loss * 0.2

simulated_portfolio_losses = np.random.normal(loc=mean_loss * len(df), scale=std_loss * np.sqrt(len(df)), size=num_simulations)
var_99 = np.percentile(simulated_portfolio_losses, 99)

print(f"Portfolio VaR (99%): {var_99:,.2f}")